# GPF Response Attribution

Explore the recorded results of a response-only, closed-set attribution experiment: twelve model labels or six developer families. Start with shipped aggregate metrics; no dataset download or training occurs automatically.

These are not human-versus-AI detection scores, authenticated checkpoint predictions, or model-quality rankings. Read [the report](../docs/REPORT.md) and [model card](../docs/MODEL_CARD.md) before using predictions.


In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "config/experiment.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config/experiment.json").exists(), "Open this notebook inside the repository."
RESULTS = ROOT / "results"
metrics = pd.read_csv(RESULTS / "metrics.csv", dtype={"fold": str})
pooled = metrics[metrics["fold"] == "pooled"].copy()
pooled[["target", "population", "split_strategy", "representation", "n_test", "accuracy", "macro_f1", "balanced_accuracy"]]


## Select an evaluated condition

`identity` holds all responses for selected identity labels out of training. `article` trains on one article and tests on the other. `family` holds out one question objective. These answer different transfer questions. `screened` removes five complete suspect cells and contains 6,996 rows.


In [ ]:
TARGET = "model"          # "model" or "company"
POPULATION = "full"       # "full" or "screened"
SPLIT = "identity"        # "identity", "article", or "family"
REPRESENTATION = "char_svm"
selection = pooled[
    (pooled.target == TARGET) & (pooled.population == POPULATION)
    & (pooled.split_strategy == SPLIT) & (pooled.representation == REPRESENTATION)
]
assert len(selection) == 1, "This combination was not evaluated; inspect the table above."
experiment_id = selection.iloc[0].experiment_id
selection


In [ ]:
folds = metrics[(metrics.experiment_id == experiment_id) & (metrics.fold != "pooled")]
folds[["fold", "n_train", "n_test", "accuracy", "macro_f1", "balanced_accuracy"]]


## Per-class performance and confusion

Company sizes differ: Google supplies three streams, four companies supply two each, and xAI supplies one. Accuracy alone can obscure class imbalance. SVM margins are not calibrated probabilities.


In [ ]:
per_class = pd.read_csv(RESULTS / "per_class.csv")
selected_classes = per_class[per_class.experiment_id == experiment_id]
selected_classes.sort_values("recall")[["label", "support", "precision", "recall", "f1"]]


In [ ]:
confusion = pd.read_csv(RESULTS / "confusion_counts.csv")
selected_confusion = confusion[confusion.experiment_id == experiment_id]
counts = selected_confusion.pivot(index="true_label", columns="predicted_label", values="count").fillna(0).astype(int)
row_percent = counts.div(counts.sum(axis=1), axis=0).mul(100).round(1)
row_percent


In [ ]:
selected_confusion[selected_confusion.true_label != selected_confusion.predicted_label].sort_values("count", ascending=False).head(12)[["true_label", "predicted_label", "count"]]


## Inspect saved held-out predictions

The compressed file contains row/group identifiers, true/predicted labels, and scores, without the original response text. A held-out prediction must remain separate from a later final model fitted on all data.


In [ ]:
predictions_path = RESULTS / "predictions.csv.gz"
if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    selected_predictions = predictions[predictions.experiment_id == experiment_id]
    display(selected_predictions.head())
else:
    print("Download the complete results bundle to inspect row-level predictions.")


## Optional source download and retraining

The source CSV has mixed third-party rights and is not bundled in this repository. The downloader uses a pinned source and validates the decompressed SHA-256. These actions are opt-in: switch the flags only if you want to download the data or rerun the experiment. Install the exact `requirements.txt` dependencies in Python 3.12 first.


In [ ]:
RUN_DOWNLOAD = False
RUN_TRAINING = False
if RUN_DOWNLOAD:
    subprocess.run([sys.executable, "scripts/download_data.py", "--output", "data/responses.csv"], cwd=ROOT, check=True)
if RUN_TRAINING:
    subprocess.run([sys.executable, "scripts/train.py", "--data", "data/responses.csv", "--output", "results"], cwd=ROOT, check=True)
    subprocess.run([sys.executable, "scripts/build_dashboard.py"], cwd=ROOT, check=True)


## Optional prediction demonstration

Run only with a trusted locally trained artifact. A closed-set system will choose among known labels even for human text, unknown models, edited responses, or unrelated topics. Do not interpret this demonstration as a validated provenance judgment. Do not use a training-corpus response to estimate test performance of the final all-data model.


In [ ]:
RUN_DEMO = False
response_text = ""  # Supply response text only, never its known label or original prompt.
artifact = ROOT / "artifacts/model_char_full.joblib"
if RUN_DEMO:
    assert response_text.strip(), "Provide a response first."
    assert artifact.exists(), "Train the final local artifact first."
    sys.path.insert(0, str(ROOT))
    from gpf_attribution.inference import predict_text
    display(predict_text(str(artifact), response_text, 3))


## Export a summary without altering raw data


In [ ]:
EXPORT = False
if EXPORT:
    export_dir = ROOT / "notebook_outputs"
    export_dir.mkdir(exist_ok=True)
    selection.to_csv(export_dir / "selected_experiment.csv", index=False)
    selected_classes.to_csv(export_dir / "selected_per_class.csv", index=False)
    row_percent.to_csv(export_dir / "selected_confusion_percent.csv")


## Citation

Haque, Mohd Ariful, and Kishor Datta Gupta. 2026. *GPF Matched-News Dataset: Semantic Variation Across Models and Identity Prompts*. [ResearchGate preprint](https://www.researchgate.net/publication/414852735_GPF_Matched-News_Dataset_Semantic_Variation_Across_Models_and_Identity_Prompts).

[Original dataset](https://github.com/kishordgupta/gpf-semantic-response-audit) · [Attribution code](https://github.com/kishordgupta/gpf-response-attribution) · [Rights notice](../RIGHTS.md)
